## 세포라 top10 수집및 모든리뷰 2년치 수집코드

In [2]:
!pip install keybert

In [1]:
import requests
import json
import time
import random
from datetime import datetime, timedelta

# ── 설정 ──────────────────────────────────────────────────────────
BV_PASSKEY       = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
BV_API_URL       = "https://api.bazaarvoice.com/data/reviews.json"
RANK_SAVE_FILE   = "sephora_rankings_current.jsonl"   
REVIEW_SAVE_FILE = "sephora_reviews_master.jsonl"   

# [날짜 설정] 현재 날짜 기준 정확히 2년 전 날짜 계산
CUTOFF_DATE = datetime.now() - timedelta(days=365 * 2)

SEPHORA_API_URL = (
    "https://www.sephora.com/api/v2/catalog/categories/skincare/seo"
    "?targetSearchEngine=NLP"
    "&sortBy=P_BEST_SELLING%3A1%3A%3AP_RATING%3A1%3A%3AP_PROD_NAME%3A0"
    "&currentPage=1&pageSize=60&content=true"
    "&includeRegionsMap=true&pickupRampup=true&sddRampup=true"
    "&includeEDD=true&loc=en-US&ch=rwd&user-segment=external-app"
)
HEADERS = {
    "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept"         : "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer"        : "https://www.sephora.com/shop/skincare?sortBy=BEST_SELLING",
    "Origin"         : "https://www.sephora.com",
}


# ── 함수: 단일 상품 리뷰 수집 (최근 2년치 필터링) ───────────────────
def get_sephora_reviews_2years(product_id, product_name):
    all_reviews = []
    limit  = 100
    offset = 0
    total  = None
    
    # 루프 중단을 위한 플래그
    stop_collecting = False

    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 (기준: {CUTOFF_DATE.strftime('%Y-%m-%d')} 이후)")

    while not stop_collecting:
        params = {
            "Filter"    : ["contentlocale:en*", f"ProductId:{product_id}"],
            "Sort"      : "SubmissionTime:desc",  # 최신순 정렬 필수
            "Limit"     : limit,
            "Offset"    : offset,
            "Include"   : "Products,Comments",
            "Stats"     : "Reviews",
            "passkey"   : BV_PASSKEY,
            "apiversion": "5.4",
            "Locale"    : "en_US",
        }
        try:
            response = requests.get(BV_API_URL, params=params, timeout=15)
            data     = response.json()

            if total is None:
                total = data.get("TotalResults", 0)
                print(f"  📊 상품 전체 리뷰 수: {total}개")

            results = data.get("Results", [])
            if not results:
                break

            for rev in results:
                # 1. 날짜 파싱 (예: 2026-03-14T18:33:38.000+00:00)
                sub_time_str = rev.get("SubmissionTime")
                if sub_time_str:
                    # ISO 형식 문자열을 datetime 객체로 변환 (앞 19자리 'YYYY-MM-DDTHH:MM:SS'만 사용)
                    sub_time = datetime.strptime(sub_time_str[:19], "%Y-%m-%dT%H:%M:%S")
                    
                    # 2. 날짜 비교: 기준일(2년 전)보다 과거이면 즉시 중단
                    if sub_time < CUTOFF_DATE:
                        stop_collecting = True
                        break
                
                # 3. 데이터 저장
                all_reviews.append({
                    "product_id"    : product_id,
                    "ReviewId"      : rev.get("Id"),
                    "Author"        : rev.get("UserNickname"),
                    "Rating"        : rev.get("Rating"),
                    "Title"         : rev.get("Title"),
                    "ReviewText"    : rev.get("ReviewText"),
                    "SubmissionTime": sub_time_str,
                    "Date"          : sub_time.strftime("%Y-%m-%d") if sub_time_str else "N/A",
                    "IsRecommended" : rev.get("IsRecommended"),
                    "SkinType"      : rev.get("ContextDataValues", {}).get("skinType", {}).get("ValueLabel", "N/A"),
                    "AgeRange"      : rev.get("ContextDataValues", {}).get("ageRange", {}).get("ValueLabel", "N/A"),
                    "Incentivized"  : rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A")
                })

            if stop_collecting:
                print(f"\n  📍 2년 이전 리뷰 도달 - 수집 중단")
                break

            offset += limit
            print(f"  🔄 분석 중... 현재 {len(all_reviews)}개 확보", end="\r")
            time.sleep(random.uniform(0.3, 0.6))

            # API 오프셋이 전체 개수를 넘으면 종료
            if offset >= total:
                break

        except Exception as e:
            print(f"\n  ❌ 에러 발생: {e}")
            break

    print(f"\n  ✨ 최종 수집 완료: {len(all_reviews)}개")
    return all_reviews


# ── STEP 1: 세포라 스킨케어 베스트셀러 Top 10 수집 ────────────────
print(f"📡 Sephora API 호출 (기준일: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
resp = requests.get(SEPHORA_API_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
data = resp.json()

products_raw = data.get("products") or data.get("catalog", {}).get("products") or []

if not products_raw:
    print("⚠️ 상품 목록을 찾지 못했습니다.")
else:
    rank_data_list    = []
    all_review_master = []
    rank_count        = 1

    for product in products_raw:
        if rank_count > 10:
            break

        # 광고/스폰서 상품 제외
        if product.get("isSponsored") or product.get("sponsored") or product.get("adBadge"):
            continue

        brand       = product.get("brandName", "N/A")
        title       = product.get("displayName", "N/A")
        sku         = product.get("currentSku", {})
        price       = sku.get("listPrice") or product.get("listPrice") or "N/A"
        rating      = float(product.get("rating", 0) or 0)
        reviews_cnt = int(product.get("reviews", 0) or 0)
        product_id  = product.get("productId", "N/A")
        url_path    = product.get("targetUrl") or product.get("url", "")
        product_url = f"https://www.sephora.com{url_path}" if url_path and not url_path.startswith("http") else url_path

        rank_data_list.append({
            "rank"         : rank_count,
            "brand"        : brand,
            "title"        : title,
            "rating"       : rating,
            "reviews"      : reviews_cnt,
            "price"        : price,
            "url"          : product_url,
            "product_id"   : product_id,
            "platform"     : "Sephora",
            "collected_at" : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
        print(f"\n📍 {rank_count}위: [{brand}] {title[:35]}")

        # ── STEP 2: 최근 2년치 리뷰 수집 실행 ─────────────────────────────
        reviews = get_sephora_reviews_2years(product_id, title)
        all_review_master.extend(reviews)

        rank_count += 1

    # ── STEP 3: JSONL 저장 ───────────────────────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    print(f"\n📊 작업 완료! 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개 저장됨")

📡 Sephora API 호출 (기준일: 2024-03-21)

📍 1위: [rhode] Glazing Milk Ceramide Facial Essenc

  🚀 [Glazing Milk Ceramide Facial Essenc] 리뷰 수집 (기준: 2024-03-21 이후)
  📊 상품 전체 리뷰 수: 1945개
  🔄 분석 중... 현재 1945개 확보
  ✨ 최종 수집 완료: 1945개

📍 2위: [rhode] Peptide Lip Tint Nourishing Glaze

  🚀 [Peptide Lip Tint Nourishing Glaze] 리뷰 수집 (기준: 2024-03-21 이후)
  📊 상품 전체 리뷰 수: 2179개


KeyboardInterrupt: 

## 팀원이 준 리뷰데이터 번역코드

In [ ]:
"""
translate_pipeline_sephora_full.py
- sephora 원본 전체 리뷰 번역
- 영어 → 한국어
"""

from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json, time, re
from tqdm import tqdm

# ── 설정 ─────────────────────────────────────────────────────
INPUT_FILE  = "./data/crawling/sephora_reviews_master.jsonl"   # 원본 전체 파일
OUTPUT_FILE = "sephora_master_translated_en_ko.jsonl"

BODY_COL    = "ReviewText"
ITEM_ID_COL = "product_id"
RATING_COL  = "Rating"

N_SAMPLE    = None   # None이면 전체, 숫자 넣으면 앞 N건만
CHUNK_SIZE  = 5
MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_SLEEP = 2.0
CHUNK_DELAY = 0.5


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 줄번호 방식 파싱
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_numbered(text: str, expected_n: int) -> list[str]:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    matches = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in matches}
    return [result.get(i + 1, "") for i in range(expected_n)]


def build_numbered(texts: list[str]) -> str:
    return "\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# jsonl 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_jsonl(path: str) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 단건 번역 (fallback)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip():
        return ""

    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if result and result.strip():
                return result.strip()
        except Exception:
            pass

        time.sleep(RETRY_SLEEP * (attempt + 1))

    return "번역실패"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 청크 번역 — 영어 → 한국어
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_chunk_en_ko(texts: list[str]) -> list[str]:
    if not texts:
        return []

    joined = build_numbered(texts)

    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source="en", target="ko").translate(joined)
            if not result:
                raise ValueError("빈 응답")

            parts = parse_numbered(result, len(texts))

            if all(p for p in parts):
                return parts

            for i, p in enumerate(parts):
                if not p:
                    parts[i] = translate_single(texts[i], "en", "ko")
            return parts

        except Exception:
            time.sleep(RETRY_SLEEP * (attempt + 1))

    return [translate_single(t, "en", "ko") for t in texts]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print(":inbox_tray: 원본 전체 데이터 로드...")
    raw_records = load_jsonl(INPUT_FILE)
    df = pd.DataFrame(raw_records)

    if N_SAMPLE:
        df = df.head(N_SAMPLE)

    print(f"   총 {len(df):,}건 / 컬럼: {list(df.columns)}")

    required_cols = [BODY_COL, ITEM_ID_COL, RATING_COL]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"필수 컬럼 없음: {missing_cols}")

    texts = df[BODY_COL].fillna("").astype(str).tolist()
    chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, len(texts), CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"\n:rocket: 전체 번역 시작 (en→ko)")
    print(f"   {len(texts):,}건 / {n_chunks}개 청크 / workers={MAX_WORKERS}")
    print(f"   CHUNK_SIZE={CHUNK_SIZE} / CHUNK_DELAY={CHUNK_DELAY}s")

    start_time = time.time()

    def run_chunk(chunk):
        ko_texts = translate_chunk_en_ko(chunk)
        time.sleep(CHUNK_DELAY)
        return ko_texts

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chunk_results = list(
            tqdm(
                executor.map(run_chunk, chunks),
                total=n_chunks,
                desc="번역 (en→ko)",
                unit="chunk"
            )
        )

    elapsed = time.time() - start_time
    print(f"\n:stopwatch: 번역 완료: {elapsed:.1f}초 ({elapsed/60:.1f}분)")

    ko_results = []
    for ko_chunk in chunk_results:
        ko_results.extend(ko_chunk)

    df["comment_ko"] = ko_results[:len(df)]

    def success_mask(col):
        return df[col].astype(str).str.strip().ne("") & df[col].ne("번역실패")

    ko_ok = success_mask("comment_ko")

    print(f"\n:bar_chart: 번역 결과")
    print(f"   성공률: {ko_ok.mean()*100:.1f}% ({ko_ok.sum():,}/{len(df):,}건)")
    print(f"   처리 속도: {len(df)/elapsed:.1f}건/초")

    print(f"\n:floppy_disk: 저장 중: {OUTPUT_FILE}")
    output_records = json.loads(df.to_json(orient="records", force_ascii=False))
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for record in output_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f":white_check_mark: 저장 완료: {OUTPUT_FILE}")

    print("\n:clipboard: 샘플 3건:")
    for _, row in df.head(3).iterrows():
        print(f"   product_id : {row.get(ITEM_ID_COL, 'N/A')}")
        print(f"   rating     : ★{row.get(RATING_COL, 'N/A')}")
        print(f"   원문(en)   : {str(row[BODY_COL])[:60]}...")
        print(f"   한국(ko)   : {str(row['comment_ko'])[:60]}...")
        print("   " + "─" * 52)


if __name__ == "__main__":
    main()

# 세포라 번역코드

In [4]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = "./sephora_reviews_master.jsonl"
OUTPUT_FILE = "sephora_master_translated_en_ko.jsonl"

BODY_COL    = "ReviewText"  # 세포라 리뷰 본문 키
ITEM_ID_COL = "product_id"
RATING_COL  = "Rating"

N_SAMPLE    = None  # 테스트 시 숫자 입력, 전체 번역 시 None
CHUNK_SIZE  = 5     # 한 번에 묶어서 번역할 개수
MAX_WORKERS = 4     # 병렬 스레드 수 (IP 차단 위험 시 2로 낮춤)
MAX_RETRIES = 3
RETRY_SLEEP = 2.0
CHUNK_DELAY = 0.3   # 영어-한국어는 속도가 빨라 딜레이를 약간 줄임

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 유틸리티 로직 (번호 매기기 방식)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    """리뷰 리스트를 [1] 문장1 \n [2] 문장2 형태로 조립"""
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    """번역된 텍스트에서 [1], [2] 패턴을 찾아 리스트로 분리"""
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 번역 엔진
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    """청크 실패 시 사용하는 개별 번역 함수"""
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def translate_chunk_en_ko(texts: list) -> list:
    """영어 -> 한국어 청크 번역 실행"""
    if not texts: return []
    joined = build_numbered(texts)
    
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source="en", target="ko").translate(joined)
            if not result: raise ValueError("빈 응답")
            
            parts = parse_numbered(result, len(texts))
            # 모든 파트가 정상적으로 분리되었는지 확인
            if all(p.strip() for p in parts):
                return parts
            
            # 파싱 실패 시 단건 번역으로 보충
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], "en", "ko")
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
            
    return [translate_single(t, "en", "ko") for t in texts]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 메인 파이프라인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print(f"📥 데이터 로드 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {INPUT_FILE}")
        return

    df = pd.DataFrame(records)
    if N_SAMPLE: df = df.head(N_SAMPLE)
    
    total_count = len(df)
    print(f"✅ 총 {total_count:,}건 로드 완료")

    # 데이터 준비
    texts = df[BODY_COL].fillna("").astype(str).tolist()
    chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, total_count, CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"🚀 병렬 번역 시작 (스레드={MAX_WORKERS}, 청크={CHUNK_SIZE})")
    start_time = time.time()

    def run_chunk(chunk):
        res = translate_chunk_en_ko(chunk)
        time.sleep(CHUNK_DELAY)
        return res

    all_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # tqdm 진행바와 함께 병렬 실행
        chunk_results = list(tqdm(
            executor.map(run_chunk, chunks),
            total=n_chunks,
            desc="Sephora 번역 중",
            unit="chunk"
        ))

    # 결과 병합
    for chunk in chunk_results:
        all_ko.extend(chunk)

    df['ReviewText_ko'] = all_ko[:total_count]
    elapsed = time.time() - start_time

    # 통계 및 저장
    success_rate = (df['ReviewText_ko'] != "번역실패").mean() * 100
    print(f"\n⏱ 번역 완료: {elapsed/60:.2f}분 소요 (성공률: {success_rate:.1f}%)")

    print(f"💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    # 샘플 출력
    print("\n📋 번역 샘플 (최근 3건):")
    for _, row in df.head(3).iterrows():
        print(f"  ★{row[RATING_COL]} | 원문: {str(row[BODY_COL])[:50]}...")
        print(f"        | 번역: {str(row['ReviewText_ko'])[:50]}...")
        print("-" * 50)

if __name__ == "__main__":
    main()

📥 데이터 로드 중: ./sephora_reviews_master.jsonl
✅ 총 13,727건 로드 완료
🚀 병렬 번역 시작 (스레드=4, 청크=5)


Sephora 번역 중: 100%|██████████| 2746/2746 [17:55<00:00,  2.55chunk/s]



⏱ 번역 완료: 17.92분 소요 (성공률: 100.0%)
💾 결과 저장 중: sephora_master_translated_en_ko.jsonl

📋 번역 샘플 (최근 3건):
  ★5 | 원문: It’s so good and moisturizing that sometimes I use...
        | 번역: 보습력도 좋고 너무 좋아서 가끔 보습제로 사용하고 있어요. 다른 것은 필요하지 않습니다. ...
--------------------------------------------------
  ★5 | 원문: I’m genuinely OBSESSED with this glazing milk 🤍 Th...
        | 번역: 저는 이 글레이징 밀크에 진심으로 빠져있습니다 🤍 질감이 전부예요. 너무 가볍고 부드러우며...
--------------------------------------------------
  ★1 | 원문: I had such high hopes for this product and was ver...
        | 번역: 저는 이 제품에 대해 큰 기대를 갖고 있었는데 결과에 매우 실망했습니다. 메이크업 전 베이...
--------------------------------------------------


# 키버트 키워드 분류, gpt 분류

In [3]:
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT
from openai import OpenAI

# =========================
# 설정
# =========================
INPUT_FILE = "./sephora_master_translated_en_ko.jsonl"
OUTPUT_CSV = "sephora_keybert_gpt_categorized.csv"
OUTPUT_JSONL = "sephora_keybert_gpt_categorized.jsonl"

TEXT_COL = "ReviewText_ko"
MODEL_GPT = "gpt-4o-mini"

TOP_N_KEYWORDS = 5
GPT_BATCH_SIZE = 30
SAVE_EVERY = 500

CATEGORIES = [
    "효과_성분",
    "사용감_텍스처",
    "향_냄새",
    "피부_트러블_부작용",
    "포장_배송",
    "가격_가성비",
    "고객서비스",
    "제품불량",
    "재구매_추천",
    "커버력_색상",
    "지속력_밀착력",
    "미분류"
]

# =========================
# 모델 로드
# =========================
client = OpenAI()
kw_model = KeyBERT("all-MiniLM-L6-v2")

# =========================
# jsonl 로드
# =========================
def load_jsonl(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# =========================
# 약한 전처리
# =========================
def clean_light(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================
# KeyBERT 키워드 추출
# =========================
def extract_keywords(text: str, top_n: int = 5) -> list[str]:
    text = str(text).strip()
    if not text:
        return []

    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words=None,
            top_n=top_n,
            use_mmr=True,
            diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

# =========================
# GPT 응답 파싱
# =========================
def strip_code_fence(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

# =========================
# GPT 배치 카테고리 분류
# =========================
def classify_batch_with_gpt(batch_items: list[dict]) -> list[dict]:
    """
    batch_items:
    [
      {"idx": 0, "keywords": [...], "text": "..."},
      ...
    ]
    """
    prompt_items = []
    for item in batch_items:
        prompt_items.append({
            "idx": item["idx"],
            "keywords": item["keywords"],
            "text": item["text"][:300]
        })

    prompt = f"""
다음 화장품 리뷰들을 카테고리로 분류해줘.

가능한 카테고리:
- 효과_성분
- 사용감_텍스처
- 향_냄새
- 피부_트러블_부작용
- 포장_배송
- 가격_가성비
- 고객서비스
- 제품불량
- 재구매_추천
- 미분류

규칙:
1. primary_category는 가장 핵심인 카테고리 1개
2. categories는 관련 카테고리 최대 3개
3. 반드시 위 목록 안에서만 선택
4. 애매하면 미분류
5. 반드시 JSON만 출력

입력:
{json.dumps(prompt_items, ensure_ascii=False, indent=2)}

출력 형식:
{{
  "results": [
    {{
      "idx": 0,
      "primary_category": "",
      "categories": ["", ""],
      "reason": ""
    }}
  ]
}}
"""

    try:
        res = client.chat.completions.create(
            model=MODEL_GPT,
            temperature=0,
            messages=[
                {"role": "system", "content": "너는 화장품 리뷰 분류기다. 반드시 JSON만 출력해라."},
                {"role": "user", "content": prompt}
            ]
        )

        raw = strip_code_fence(res.choices[0].message.content)
        parsed = json.loads(raw)
        results = parsed.get("results", [])

        # 방어적 후처리
        cleaned_results = []
        for r in results:
            primary = r.get("primary_category", "미분류")
            cats = r.get("categories", ["미분류"])
            reason = r.get("reason", "")

            if primary not in CATEGORIES:
                primary = "미분류"

            cats = [c for c in cats if c in CATEGORIES]
            if not cats:
                cats = [primary]

            cleaned_results.append({
                "idx": r.get("idx"),
                "primary_category": primary,
                "categories": cats[:3],
                "reason": reason
            })

        return cleaned_results

    except Exception as e:
        # 실패 시 전부 미분류
        return [
            {
                "idx": item["idx"],
                "primary_category": "미분류",
                "categories": ["미분류"],
                "reason": f"GPT 분류 실패: {repr(e)}"
            }
            for item in batch_items
        ]

# =========================
# 데이터 로드
# =========================
df = load_jsonl(INPUT_FILE)

print("총 데이터 수:", len(df))
print("컬럼:", df.columns.tolist())

df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].copy()
df = df.reset_index(drop=True)

df["text_for_model"] = df[TEXT_COL].apply(clean_light)

print("유효 데이터 수:", len(df))

# =========================
# 1단계: KeyBERT 키워드 추출
# =========================
keywords_list = []

for text in tqdm(df["text_for_model"], total=len(df), desc="KeyBERT 키워드 추출"):
    kws = extract_keywords(text, top_n=TOP_N_KEYWORDS)
    keywords_list.append(kws)

df["keybert_keywords"] = keywords_list

# =========================
# 2단계: GPT 배치 분류
# =========================
primary_categories = ["미분류"] * len(df)
categories_list = [["미분류"]] * len(df)
reasons = [""] * len(df)

batch = []
processed = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="GPT 배치 분류"):
    batch.append({
        "idx": idx,
        "keywords": row["keybert_keywords"],
        "text": row["text_for_model"]
    })

    if len(batch) == GPT_BATCH_SIZE:
        results = classify_batch_with_gpt(batch)

        for r in results:
            i = r["idx"]
            primary_categories[i] = r["primary_category"]
            categories_list[i] = r["categories"]
            reasons[i] = r["reason"]

        processed += len(batch)
        batch = []

        if processed % SAVE_EVERY == 0:
            temp_df = df.copy()
            temp_df["primary_category"] = primary_categories
            temp_df["categories"] = categories_list
            temp_df["category_reason"] = reasons
            temp_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
            print(f"\n중간 저장 완료: {processed}건")

        time.sleep(0.5)

# 마지막 배치 처리
if batch:
    results = classify_batch_with_gpt(batch)

    for r in results:
        i = r["idx"]
        primary_categories[i] = r["primary_category"]
        categories_list[i] = r["categories"]
        reasons[i] = r["reason"]

# =========================
# 결과 반영
# =========================
df["primary_category"] = primary_categories
df["categories"] = categories_list
df["category_reason"] = reasons

# =========================
# 저장
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print("\n:white_check_mark: 완료")
print("CSV:", OUTPUT_CSV)
print("JSONL:", OUTPUT_JSONL)

print("\n1차 카테고리 분포:")
print(df["primary_category"].value_counts())

print("\n샘플:")
print(df[[TEXT_COL, "keybert_keywords", "primary_category", "categories"]].head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


총 데이터 수: 13727
컬럼: ['product_id', 'ReviewId', 'Author', 'Rating', 'Title', 'ReviewText', 'SubmissionTime', 'Date', 'IsRecommended', 'SkinType', 'AgeRange', 'Incentivized', 'ReviewText_ko']
유효 데이터 수: 13703


GPT 배치 분류:   2%|▏         | 209/13703 [03:50<4:07:43,  1.10s/it]


KeyboardInterrupt: 